# Agentic AI & RAG Engineering — Course Notebook (Weeks 1–3)

**Author:** Narayanan Palani  
**Scope:** Weeks 1 to 3 Complete Implementation Reference  
**Stack:** Python, Ollama (`gpt-oss:20b`), Pydantic v2, AsyncIO, SQLite, FastAPI, Streamlit, Pytest

--- 
## 1. Environment Setup & Dependency Configuration
* **Slide Source:** *Week 1 - Slide 17 ("Setup — Two Minutes"), Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")*
* **Action:** Install required packages and securely load API credentials using `python-dotenv`.

In [1]:
# Week 1 - Hello LLM
# Purpose: Run a first local LLM request through Ollama.
#
# One-time setup:
#   1. Install Ollama from https://ollama.com using code at cmd prompt: curl -fsSL https://ollama.com/install.sh | sh
#   2. Start the Ollama service in an exclusive command prompt: ollama run gpt-oss:20b
#   3. Pull the local model from Terminal:
#        ollama pull gpt-oss:20b
#
# Ollama runs the model locally, so this inference does not use OpenAI API credits
# or require an OPENAI_API_KEY.

# Install the Ollama Python package in this notebook environment.
# Run this once if the package is not already installed.
%pip install -q ollama

# Import the Ollama Python client.
import ollama

# Send a chat request to the local Ollama model.
response = ollama.chat(
    # Use the same local model throughout this notebook.
    model="gpt-oss:20b",

    # Send only the user message needed for this experiment.
    messages=[
        {
            "role": "user",
            "content": "What is the core benefit of RAG? Answer with exactly one word."
        }
    ]
)

# Print the text generated by the local model.
print(response.message.content)


Note: you may need to restart the kernel to use updated packages.
Recall


In [2]:
# Slide Source:
#   Week 1 - Slide 17 ("Setup — Two Minutes")
#   Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")
#
# Purpose:
#   1. Verify that the Ollama Python package is available.
#   2. Confirm that the local Ollama service is reachable.
#   3. Confirm that the selected local model is installed.
#
# Local-model principle:
#   Ollama runs inference on your machine.
#   No OPENAI_API_KEY is required for these exercises.
#   No OpenAI API credits are consumed by local inference calls.
#
# Before running this cell:
#   1. Install Ollama from https://ollama.com
#   2. Start the Ollama service.
#   3. Pull the model from Terminal:
#        ollama pull gpt-oss:20b

import ollama

# Query the local Ollama service for installed models.
# This checks connectivity without generating a model response.
models = ollama.list()

# Extract the local model names.
installed_models = [model.model for model in models.models]

# Stop early with a clear instruction if the required model is missing.
if not any(name == "gpt-oss:20b" for name in installed_models):
    raise RuntimeError(
        f"Local Ollama model 'gpt-oss:20b' was not found. "
        f"Run: ollama pull gpt-oss:20b"
    )

# Confirm that the local environment is ready.
print("Local Ollama environment successfully initialized.")
print(f"Model available: gpt-oss:20b")


Local Ollama environment successfully initialized.
Model available: gpt-oss:20b


--- 
## 2. Week 1: Foundations, Decision Frameworks & Hello LLM
* **Slide Source:** *Week 1 - Slide 06 ("A Working Definition"), Slide 11-16 ("Four Patterns"), Slide 17-18 ("Hello LLM & Lab Step 2")*
* **Action:** Initialize a local Ollama client, run a baseline prompt using `gpt-oss:20b`, and log local inference metrics.

In [3]:
# Local Ollama connectivity check
#
# This replaces a cloud-model/API connectivity check.
# It verifies that:
#   1. The Ollama service is running locally.
#   2. The Python client can communicate with it.
#   3. Local models are visible to the client.
#
# No LLM prompt is generated by this check.

import ollama

# Ask the local Ollama service for its installed model list.
models = ollama.list()

print("Ollama service is reachable.")
print("Installed local models:")

# Display the local models available to this notebook.
for model in models.models:
    print(f"- {model.model}")


Ollama service is reachable.
Installed local models:
- gpt-oss:20b


In [4]:
# Week 1 - Hello LLM / Lab Step 2
#
# This version uses Ollama instead of the local Ollama service.
# The model runs locally, so there is no API key or API credit requirement.

# Import the Ollama Python client.
import ollama

# Use one model consistently throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"


def run_hello_llm(prompt_text: str) -> str:
    """
    Send a prompt to the local Ollama model and return its response.

    prompt_text:
        The question/instruction sent to the local model.

    Returns:
        The text generated by the local model.
    """

    # Send the prompt to Ollama running on the local machine.
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        options={
            # Keep generation focused and relatively deterministic.
            "temperature": 0.2,
            # Limit generation for this one-word experiment.
            "num_predict": 1000
        }
    )

    # Ollama reports local inference statistics in the response.
    # prompt_eval_count = input tokens evaluated locally.
    # eval_count = output tokens generated locally.
    prompt_tokens = getattr(response, "prompt_eval_count", 0)
    completion_tokens = getattr(response, "eval_count", 0)

    # Calculate total locally processed tokens for visibility.
    total_tokens = prompt_tokens + completion_tokens

    print(
        f"Prompt tokens: {prompt_tokens} | "
        f"Completion tokens: {completion_tokens} | "
        f"Total tokens: {total_tokens}"
    )

    # Extract and return only the generated text.
    return response.message.content


# Keep the prompt short to demonstrate prompt-efficiency principles.
prompt = "What is the core benefit of RAG? Answer with exactly one word."

# Run the local model.
result = run_hello_llm(prompt)

# Display the generated response.
print("\nLLM Response:")
print(result)


Prompt tokens: 82 | Completion tokens: 254 | Total tokens: 336

LLM Response:
Accuracy


--- 
## 3. Week 2: Typed Contracts (Pydantic) & Async Concurrency Pipeline
* **Slide Source:** *Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"), Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")*
* **Action:** Build Pydantic schemas, an async client with exponential retry backoff, parallel batch processing via `asyncio.gather`, JSON structured logging, and SQLite persistence.

In [5]:
# Slide Source: Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"),
# Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")
#
# Action: Build Pydantic schemas, an async Ollama client with exponential retry
# backoff, parallel batch processing via asyncio.gather, JSON structured logging,
# and SQLite persistence.
#
# Ollama replaces the paid cloud API for this notebook. The model runs locally.

import asyncio
import json
import logging
import sqlite3
import time
from typing import List
from ollama import AsyncClient
from pydantic import BaseModel, Field

# Use the same local model throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"

# Global lock to serialize database writes across concurrent tasks
db_lock = asyncio.Lock()

# Structured Logging Setup
logging.basicConfig(level=logging.INFO, format="%(message)s")


def log_json(event: str, **kwargs):
    log_entry = {"event": event, "timestamp": time.time(), **kwargs}
    logging.info(json.dumps(log_entry))


# Pydantic Schemas
class QuestionRequest(BaseModel):
    id: int
    query: str = Field(..., min_length=3, description="The user query")


class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str
    status: str = "success"


# SQLite Persistence Initialization
def init_db():
    # Use timeout to wait up to 20 seconds for lock release if database is busy
    conn = sqlite3.connect("results.db", timeout=20.0)
    cursor = conn.cursor()

    # Enable Write-Ahead Logging (WAL) mode for drastically better concurrency
    cursor.execute("PRAGMA journal_mode=WAL;")

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS run_results (
            id INTEGER PRIMARY KEY,
            query TEXT NOT NULL,
            answer TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """
    )

    conn.commit()
    conn.close()


def _sync_save_to_db(record: AnswerResponse):
    """Synchronous SQLite write operation with extended timeout."""
    conn = sqlite3.connect("results.db", timeout=20.0)
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO run_results (id, query, answer) VALUES (?, ?, ?)",
        (record.id, record.query, record.answer),
    )

    conn.commit()
    conn.close()


async def save_to_db(record: AnswerResponse):
    """Thread-safe, non-blocking async wrapper around DB writes."""
    async with db_lock:
        # Offload blocking SQLite I/O to a worker thread
        await asyncio.to_thread(_sync_save_to_db, record)


# Retry Wrapper with Exponential Backoff
async def with_retry(coro_func, *args, max_retries: int = 3, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            return await coro_func(*args, **kwargs)
        except Exception as exc:
            log_json(
                "async_retry_attempt",
                attempt=attempt,
                max_retries=max_retries,
                error=str(exc),
            )

            if attempt == max_retries:
                raise

            await asyncio.sleep(2**attempt)


async def process_single_query(
    async_client: AsyncClient, req: QuestionRequest
) -> AnswerResponse:
    async def _call():
        response = await async_client.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": req.query}],
            options={"temperature": 0.3},
        )

        # Extract message content safely
        if hasattr(response, "message"):
            return response.message.content.strip()
        return response["message"]["content"].strip()

    answer_text = await with_retry(_call)

    res = AnswerResponse(id=req.id, query=req.query, answer=answer_text)

    # Safely save to DB asynchronously without lock contention
    await save_to_db(res)

    log_json("query_processed", id=req.id, query=req.query)
    return res


async def batch_process_pipeline(queries: List[QuestionRequest]):
    init_db()
    async_client = AsyncClient()

    tasks = [process_single_query(async_client, q) for q in queries]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results


# Driver Code Execution
async def main():
    sample_queries = [
        QuestionRequest(id=1, query="What is Pydantic in Python?"),
        QuestionRequest(
            id=2, query="How does asyncio.gather enable concurrent local model calls?"
        ),
        QuestionRequest(
            id=3, query="Why use SQLite for local execution persistence?"
        ),
    ]

    print("Starting Async Local Ollama Batch Pipeline...")
    batch_results = await batch_process_pipeline(sample_queries)

    for res in batch_results:
        if isinstance(res, Exception):
            print(f"\nBatch item failed: {res}")
        else:
            print(
                f"\nID: {res.id}"
                f"\nQuery: {res.query}"
                f"\nAnswer: {res.answer[:100]}..."
            )


if __name__ == "__main__":
    if "get_ipython" in globals():
        import nest_asyncio

        nest_asyncio.apply()
        asyncio.run(main())
    else:
        asyncio.run(main())

Starting Async Local Ollama Batch Pipeline...


OperationalError: database is locked

--- 
## 4. Week 3: FastAPI Web Service & Streaming Endpoint
* **Slide Source:** *Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")*
* **Action:** Write `main.py` containing FastAPI backend supporting a health probe, structured POST endpoint, and streaming token response via SSE / raw stream.

In [ ]:
%%writefile main.py
# Slide Source: Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")
#
# Action: Build FastAPI backend supporting a health probe, a structured POST
# endpoint, and a streaming response backed by a local Ollama model.
#
# Run locally with:
#   uvicorn main:app --reload --port 8000

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from ollama import AsyncClient

# The local Ollama model used by this backend.
OLLAMA_MODEL = "gpt-oss:20b"

# Create the FastAPI application.
app = FastAPI(
    title="Agentic RAG Engine API",
    version="2.0.0"
)

# Create one asynchronous Ollama client.
# Ollama communicates with the local Ollama service.
async_client = AsyncClient()


class AskRequest(BaseModel):
    # Text supplied by the API caller.
    query: str


@app.get("/health")
async def health_check():
    # Lightweight application health probe.
    # This does not invoke the LLM.
    return {
        "status": "ok",
        "service": "agentic-rag-engine",
        "model": OLLAMA_MODEL
    }


@app.post("/ask")
async def ask_endpoint(request: AskRequest):
    # Reject empty or whitespace-only queries.
    if not request.query.strip():
        raise HTTPException(
            status_code=400,
            detail="Query string cannot be empty."
        )

    # Send the request to the local Ollama model.
    response = await async_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": request.query}
        ]
    )

    # Return the original query and generated answer as JSON.
    return {
        "query": request.query,
        "answer": response.message.content
    }


async def stream_generator(query: str):
    # Request streaming generation from the local Ollama model.
    response = await async_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": query}
        ],
        stream=True
    )

    # Yield generated text chunks as they arrive.
    async for chunk in response:
        content = chunk.message.content
        if content:
            yield content


@app.post("/stream")
async def stream_endpoint(request: AskRequest):
    # Return a streaming HTTP response to the caller.
    return StreamingResponse(
        stream_generator(request.query),
        media_type="text/plain"
    )


--- 
## 5. Streamlit Frontend UI
* **Slide Source:** *Week 3 - Slide 16 ("Minimal Streamlit UI")*
* **Action:** Write `app.py` constructing UI consuming the FastAPI streaming response in real-time.

In [ ]:
%%writefile app.py
# Slide Source: Week 3 - Slide 16 ("Minimal Streamlit UI")
# Action: Construct UI consuming the FastAPI streaming response in real-time.

import streamlit as st
import requests

st.set_page_config(page_title="Agentic RAG Control Center", layout="wide")
st.title("Agentic AI & RAG Interface")

query_input = st.text_input("Enter your request or prompt:", placeholder="Ask something...")

if st.button("Submit Query"):
    if not query_input.strip():
        st.warning("Please enter a valid query.")
    else:
        st.subheader("Streaming Response:")
        response_box = st.empty()
        full_response = ""
        
        try:
            url = "http://localhost:8000/stream"
            with requests.post(url, json={"query": query_input}, stream=True) as response:
                if response.status_code == 200:
                    for chunk in response.iter_content(chunk_size=1024, decode_unicode=True):
                        if chunk:
                            full_response += chunk
                            response_box.markdown(full_response + "▌")
                    response_box.markdown(full_response)
                else:
                    st.error(f"Error {response.status_code}: Unable to reach API.")
        except Exception as e:
            st.error(f"Connection error: {str(e)}")

# Run with command: streamlit run app.py

--- 
## 6. Week 3 (Day 2): Testing, Mocks & API Contract Specification
* **Slide Source:** *Week 3 - Slide 28-30 ("Testing & Mocks"), Slide 33-36 ("API Contracts & ADR 0002")*
* **Action:** Unit test pipeline components with `pytest` & `AsyncMock`, and document Architectural Decision Record (ADR 0002).

In [3]:
# Slide Source: Week 3 - Slide 28-30 ("Testing & Mocks")
#
# Action: Test execution components without making real local-model calls.
# AsyncMock simulates the Ollama AsyncClient response.
#
# The test deliberately avoids inference, so it is free and fast.

import pytest
from unittest.mock import AsyncMock
from pydantic import BaseModel


class QuestionRequest(BaseModel):
    id: int
    query: str


class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str


async def dummy_llm_call(query: str, client) -> str:
    # Call the same method shape used by the production Ollama client.
    response = await client.chat(
        model="gpt-oss:20b",
        messages=[
            {"role": "user", "content": query}
        ]
    )

    # Return the generated message text.
    return response.message.content


@pytest.mark.asyncio
async def test_dummy_llm_call_success():
    # Create a fake asynchronous Ollama client.
    mock_client = AsyncMock()

    # Build a fake response object with the structure expected
    # by dummy_llm_call().
    mock_message = AsyncMock()
    mock_message.content = "Mocked answer payload"

    mock_completion = AsyncMock()
    mock_completion.message = mock_message

    # Configure the fake client's chat() call to return our mock response.
    mock_client.chat.return_value = mock_completion

    # Execute the function under test.
    result = await dummy_llm_call(
        "Test Query",
        mock_client
    )

    # Verify the returned answer.
    assert result == "Mocked answer payload"

    # Verify that the local-model client was called exactly once.
    mock_client.chat.assert_called_once()

    print("Test passed successfully!")


# Run the asynchronous test directly in the notebook.
await test_dummy_llm_call_success()


Test passed successfully!


### ADR 0002: API Interface Contract Locking

**Context:** Standardizing the interface contract for client applications interacting with the RAG microservice.

**Decision:** All standard interactions will adhere strictly to Pydantic JSON validation schemas.

#### Schema Spec: `/v1/ask`
* **Request (POST):** `{"query": "string (min_length: 3)"}`
* **Response (200 OK):** `{"query": "string", "answer": "string"}`
* **Error Response (400 Bad Request):** `{"detail": "Query string cannot be empty."}`

### Building a Resilient, Asynchronous LLM Pipeline
An architectural breakdown of the Week 2 local Ollama batch processor.

The provided code represents a robust, production-ready pattern for communicating with Local LLMs (Large Language Models) in an asynchronous environment. Instead of making slow, sequential requests, this pipeline processes multiple prompts concurrently while enforcing strict data types and handling failures gracefully.

Here is a breakdown of how the different components work together:

Enforcing Strict Contracts with Pydantic
In agentic workflows, unpredictable data structures lead to broken pipelines. The code uses Pydantic BaseModel classes (Question and Answer) to create strict data contracts. This ensures that every piece of data moving through the pipeline conforms to an expected format, reducing the risk of runtime errors when passing data between the UI, the LLM, and the database.

The Async LLM Engine
The async_ask_ollama function is the core engine. It swaps out the original lab's fake sleep timers for genuine, asynchronous HTTP calls to a local Ollama model (gpt-oss:20b) using ollama.AsyncClient().

To effectively test the resilience of the system, this function also features a deliberate fail_rate parameter. By artificially triggering TransientError exceptions, developers can simulate real-world network hiccups, API rate limits, or locked databases without needing to actually break their local server.

Resilience via Exponential Backoff
AI generation can be resource-intensive, and requests occasionally time out or fail. The ask_llm_with_retry wrapper prevents a single failed request from crashing the entire batch process.

It catches simulated transient failures.

It pauses before trying again using an exponential backoff strategy (waiting 1 second, then 2 seconds, then 4 seconds).

This prevents overwhelming an already struggling server with immediate, repeated requests.

Concurrency Pattern: Batch vs. Stream
The code provides two distinct ways to handle concurrent LLM requests, both utilizing Python's asyncio library:

Strict Ordering (run_batch): Uses asyncio.gather to fire off all questions simultaneously, but waits for the absolute slowest response to finish before returning anything. The output strictly matches the order of the input.

Speed to Insight (run_batch_stream): Uses asyncio.as_completed. As soon as any single LLM call finishes—regardless of its position in the original list—it is immediately processed and printed. This is crucial for user-facing applications (like Streamlit UIs) where you want to show progress immediately rather than making the user stare at a loading screen until the entire batch is done.

In [7]:
# Week 2 - Typed Contracts & Async Concurrency Pipeline
import asyncio
import random
from pydantic import BaseModel
from ollama import AsyncClient

# ─── Typed Contracts (Pydantic) ─────────────────────────────────────────

class Question(BaseModel):
    text: str

class Answer(BaseModel):
    text: str


# ─── Real Async LLM Call (Ollama) ───────────────────────────────────────

class TransientError(Exception):
    """Raised to simulate transient network failures for testing retries."""
    pass

async def async_ask_ollama(question: Question, fail_rate: float = 0.0) -> Answer:
    """
    Calls the local Ollama model asynchronously.
    Includes an artificial fail_rate to demonstrate retry logic in the lab.
    """
    if random.random() < fail_rate:
        await asyncio.sleep(0.1)  # small delay before failing
        raise TransientError("Simulated transient failure (e.g., timeout or locked DB)")
    
    # Real asynchronous call to the local LLM
    response = await AsyncClient().chat(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": question.text}]
    )
    
    return Answer(text=response['message']['content'])


# ─── Retry wrapper with exponential backoff ─────────────────────────────

async def ask_llm_with_retry(question: Question, fail_rate: float = 0.0,
                             max_attempts: int = 3) -> Answer:
    """Wraps the Ollama call with 1s/2s/4s backoff on transient failures."""
    for attempt in range(max_attempts):
        try:
            return await async_ask_ollama(question, fail_rate=fail_rate)
        except TransientError as e:
            if attempt == max_attempts - 1:
                raise
            backoff_seconds = 2 ** attempt  # 1, 2, 4
            print(f"    [Retry] Attempt {attempt + 1} failed for '{question.text[:20]}...'. Retrying in {backoff_seconds}s...")
            await asyncio.sleep(backoff_seconds)
    raise RuntimeError("unreachable")


# ─── run_batch — the lab pattern (gather, returns in input order) ───────

async def run_batch(questions: list[Question],
                     fail_rate: float = 0.0) -> list[Answer]:
    """All-or-nothing: returns when slowest call finishes, in input order."""
    tasks = [ask_llm_with_retry(q, fail_rate=fail_rate) for q in questions]
    return await asyncio.gather(*tasks)


# ─── run_batch_stream — the W2 activity pattern (as_completed) ──────────

async def run_batch_stream(questions: list[Question],
                            fail_rate: float = 0.0) -> list[Answer]:
    """Streams: prints each answer as it arrives, returns in completion order.

    Use this when you want to start processing as results stream in,
    or to see which calls are slow vs fast. Don't use this when input
    order matters in the returned list.
    """
    tasks = [ask_llm_with_retry(q, fail_rate=fail_rate) for q in questions]
    results: list[Answer] = []

    for coro in asyncio.as_completed(tasks):
        ans = await coro
        
        # Print the FULL answer the instant it's ready
        print(f"  ✓ {ans.text.strip()}\n") 
        print("-" * 60)  # Adds a dashed line to separate the long answers
        
        results.append(ans)
    return results
    # for coro in asyncio.as_completed(tasks):
    #     ans = await coro
    #     # Print a snippet of the answer the instant it's ready
    #     print(f"  ✓ {ans.text.strip()[:80]}...") 
    #     results.append(ans)
    # return results


# ─── Execution Block (Notebook-friendly) ────────────────────────────────

# In Jupyter, we are already in an event loop, so we can await directly.
fail_rate_test = 0.3  # Set to 0.0 for a clean run, > 0.0 to test retries

sample_questions = [Question(text=t) for t in [
    "What is RAG in one sentence?",
    "Name three uses of vector databases.",
    "Why might an LLM hallucinate?",
    "Explain async and await in plain language.",
    "What is the difference between a chatbot and an agent?",
]]

print(f"\nStarting Async Local Ollama Batch Pipeline (Stream Mode) — fail_rate={fail_rate_test}\n")

# Run the streaming batch execution
answers = await run_batch_stream(sample_questions, fail_rate=fail_rate_test)

print(f"\nSuccessfully returned {len(answers)} real LLM answers.")


Starting Async Local Ollama Batch Pipeline (Stream Mode) — fail_rate=0.3

    [Retry] Attempt 1 failed for 'Explain async and aw...'. Retrying in 1s...


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  ✓ **Three common uses of vector databases:**

1. **Semantic/semantic‑search engines** – Store high‑dimensional text or document embeddings so users can retrieve documents that are meaningfully related to a query, even if the exact words differ.

2. **Image / multimedia similarity search** – Keep embeddings generated from vision models; then quickly find visually similar images, videos, or audio clips by nearest‑neighbor lookup.

3. **Recommendation & personalization systems** – Use user and item embeddings to compute similarities (or distances) in real time, powering product recommendations, content suggestions, or personalized rankings.

------------------------------------------------------------


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  ✓ **RAG (Retrieval‑Augmented Generation) is a technique that first pulls relevant documents from an external knowledge base and then uses those retrieved passages to condition a generative language model, enabling it to produce more accurate, context‑rich responses.**

------------------------------------------------------------


...

  ✓ ### Why Large Language Models (LLMs) Hallucinate

In short, **LLMs hallucinate because they are pattern‑matching machines that have no built‑in mechanism to check whether the text they produce is true or grounded in reality**. The hallucination problem emerges from a combination of the way these models are trained, the data they see, and how we ask them questions.

Below is a deeper dive into the main contributors:

| # | Why it Happens | How It Manifests | Example |
|---|----------------|-----------------|---------|
| **1** | **Training Objective = Next‑Token Prediction** | The loss function simply says “pick the word most likely to come next given all previous words.” Truthfulness is never penalized; only plausibility matters. | When asked *“Who won the 2020 US election?”* a model might generate “Joe Biden” because that sequence was common in its training data, even if it has no way of verifying it. |
| **2** | **Statistical Correlations Replace Causal Knowledge** | LLMs learn *c

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  ✓ ### In a nutshell

| **Chatbot** | **Agent** |
|-------------|-----------|
| A conversational system that *talks* (text or voice) with users, usually within a single domain. | An autonomous entity—software or software+hardware—that *acts* on behalf of a user in one or more environments to achieve goals. |

---

## 1. Core Focus

| Aspect | Chatbot | Agent |
|--------|---------|-------|
| **Primary function** | Respond to queries, provide information, or entertain via dialogue. | Make decisions, take actions, and possibly learn from the environment. |
| **Goal orientation** | Often reactive: answer what the user asks. | Proactive: plan steps to reach a goal (e.g., book a flight, close a smart‑home window). |

---

## 2. Architecture & Capabilities

| Feature | Chatbot | Agent |
|---------|--------|-------|
| **State handling** | Usually stateless or simple session state; remembers only the current conversation. | Maintains long‑term memory: user preferences, past actions, world mode

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


  ✓ ## Quick‑start “plain‑language” guide

### 1. What is *async*?

- **Async** is just a way of telling the computer, “I’m going to do something that will take time (like reading a file or downloading data), and while I wait for it, I don’t want the rest of my program to freeze.”  
- In code you mark a function with `async`:

```js
async function getWeather() {
  // ...
}
```

When a function is marked async, JavaScript automatically gives it a **promise**.  
A promise is an object that represents “something will happen in the future”.

### 2. What is *await*?

- **Await** is used *inside* an async function to say: “Pause here until that promise settles (fulfills or rejects).”  
- It looks like this:

```js
const data = await fetch('/weather');
```

While JavaScript waits for the network request, it can keep running other code – it isn’t stuck.

### 3. Analogy: Making coffee while doing work

| Synchronous (normal) | Asynchronous (async/await) |
|----------------------|---------------

In [8]:
# Run the batch execution to learn the difference from streaming above
answersBatch = await run_batch(sample_questions, fail_rate=fail_rate_test)

print(f"\nSuccessfully returned {len(answersBatch)} real LLM answers.")

    [Retry] Attempt 1 failed for 'What is RAG in one s...'. Retrying in 1s...
    [Retry] Attempt 1 failed for 'Explain async and aw...'. Retrying in 1s...
    [Retry] Attempt 2 failed for 'What is RAG in one s...'. Retrying in 2s...
    [Retry] Attempt 2 failed for 'Explain async and aw...'. Retrying in 2s...


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



Successfully returned 5 real LLM answers.
